In [1]:
# import os, sys
# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# if project_root not in sys.path:
#     sys.path.append(project_root)
# from agents.excursion.agent import excursion_graph as graph
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root) 
from agents.excursion.agent import build_excursion_graph 

# Khởi tạo graph (cần dùng await vì hàm này là async)
graph = await build_excursion_graph()

c:\Program Files\Python312\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
# 1. In ra mã nguồn Mermaid của graph
mermaid_code = graph.get_graph(xray=True).draw_mermaid()
print(mermaid_code)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	excursion_chat(excursion_chat)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> excursion_chat;
	excursion_chat -.-> __end__;
	excursion_chat -.-> tools;
	tools --> excursion_chat;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [2]:
# def stream_graph_updates(inputs: dict):
#     for event in graph.stream(inputs):
#         if "excursion_chat" in event:
#             messages = event["excursion_chat"].get("messages", [])
#             for msg in messages:
#                 # Nếu là AIMessage (object)
#                 if hasattr(msg, "content") and msg.content:
#                     print("Assistant:", msg.content)
#                 # Nếu là dict
#                 elif isinstance(msg, dict) and msg.get("role") == "assistant":
#                     print("Assistant:", msg.get("content"))
#         if "tools" in event:
#             messages = event["tools"].get("messages", [])
#             for msg in messages:
#                 if hasattr(msg, "content") and msg.content:
#                     print("Tool Output:", msg.content)
#                 elif isinstance(msg, dict):
#                     print("Tool Output:", msg.get("content"))
from langchain_core.messages import AIMessage

async def stream_graph_updates(inputs: dict):
    async for event in graph.astream(inputs):
        if "excursion_chat" in event:
            for msg in event["excursion_chat"].get("messages", []):
                # In tham số tool — đây là chỗ cần check
                if isinstance(msg, AIMessage) and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print("🔧 Tool:", tc["name"])
                        print("📥 Args:", tc["args"])

                if hasattr(msg, "content") and msg.content:
                    print("Assistant:", msg.content)

        if "tools" in event:
            for msg in event["tools"].get("messages", []):
                if hasattr(msg, "content") and msg.content:
                    print("Tool Output:", msg.content)

In [3]:
test_cases = [

    "Tôi muốn kiếm trip ở Hà Nội ngày 30/8/2026."
]


for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs) 


🧑‍💻 User: Tôi muốn kiếm trip ở Hà Nội ngày 30/8/2026.
{'messages': [{'role': 'user', 'content': 'Tôi muốn kiếm trip ở Hà Nội ngày 30/8/2026.'}]}


Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 15.86502393s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 13.673310119s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 13
}
]

In [ ]:
test_cases = [

    "Tôi muốn kiếm trip ở Huế ngày 30/7/2026."
]


for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs) 


🧑‍💻 User: Tôi muốn kiếm trip ở Huế ngày 30/7/2026.
{'messages': [{'role': 'user', 'content': 'Tôi muốn kiếm trip ở Huế ngày 30/7/2026.'}]}
🔧 Tool: search_attractions_tool
📥 Args: {'end_date': '2026-07-30', 'start_date': '2026-07-30', 'location': 'Huế'}
Tool Output: [{'type': 'text', 'text': '{\n  "source": "booking_com15_rapidapi",\n  "product_id": "PRYx5yS0RtHl",\n  "external_attraction_id": "PRYx5yS0RtHl",\n  "slug": "pryx5ys0rthl-bus-transfer-from-hue-to-hoi-an-with-sightseeing",\n  "name": "Bus Transfer from Hue to Hoi An with Sightseeing",\n  "description": "This is a tour of famous landmarks from Hue to Hoi An, you will be amazed at the natural beauty o...",\n  "category": null,\n  "rating": null,\n  "review_count": null,\n  "price": 486550,\n  "currency": "VND",\n  "image": "https://r-xx.bstatic.com/xdata/images/xphoto/300x320/288455584.jpg?k=961765e926383d36c10c32ce77a4230aae187592e8055ded9ae55c778ee798c4&o=",\n  "duration": null,\n  "location": null,\n  "raw": {\n    "__typen

: 

In [ ]:
test_cases = [

    "Tôi muốn kiếm trip ở Đà Nẵng ngày 30/8/2026."
]


for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs) 


🧑‍💻 User: Tôi muốn kiếm trip ở Đà Nẵng ngày 30/6/2026.
{'messages': [{'role': 'user', 'content': 'Tôi muốn kiếm trip ở Đà Nẵng ngày 30/6/2026.'}]}
🔧 Tool: search_attractions_tool
📥 Args: {'end_date': '2026-06-30', 'start_date': '2026-06-30', 'location': 'Đà Nẵng'}
Tool Output: [{'type': 'text', 'text': '{\n  "source": "booking_com15_rapidapi",\n  "product_id": "PR9oF7hAhjy9",\n  "external_attraction_id": "PR9oF7hAhjy9",\n  "slug": "pr9of7hahjy9-sun-world-ba-na-hills-ticket",\n  "name": "Vé Sun World Ba Na Hills",\n  "description": "Cơ hội khám phá khu phức hợp nghỉ dưỡng và vui chơi giải trí Sun World Ba Na Hills",\n  "category": null,\n  "rating": null,\n  "review_count": null,\n  "price": 923130,\n  "currency": "VND",\n  "image": "https://q-xx.bstatic.com/xdata/images/xphoto/300x320/642461919.jpg?k=f5f6c2bc3f6e0ef9e24382a51f332bead7cfd96f6b36389e6663cf2056e842a0&o=",\n  "duration": null,\n  "location": null,\n  "raw": {\n    "__typename": "AttractionsProduct",\n    "cancellationPoli

In [3]:
test_cases = [

    "Tôi muốn xem pr3zaotoqy8c-morning-small-group-to-marble-mountains-am-phu-cave-monkey-mountain."
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)


🧑‍💻 User: Tôi muốn xem pr3zaotoqy8c-morning-small-group-to-marble-mountains-am-phu-cave-monkey-mountain.
{'messages': [{'role': 'user', 'content': 'Tôi muốn xem pr3zaotoqy8c-morning-small-group-to-marble-mountains-am-phu-cave-monkey-mountain.'}]}
🔧 Tool: fetch_attraction_details_tool
📥 Args: {'slug': 'pr3zaotoqy8c-morning-small-group-to-marble-mountains-am-phu-cave-monkey-mountain'}
Tool Output: [{'type': 'text', 'text': '{\n  "source": "booking_com15_rapidapi",\n  "slug": "pr3zaotoqy8c-morning-small-group-to-marble-mountains-am-phu-cave-monkey-mountain",\n  "details": {\n    "slug": "pr3zaotoqy8c-morning-small-group-to-marble-mountains-am-phu-cave-monkey-mountain",\n    "Price": 550000,\n    "Currency": "VND",\n    "AverageScore": 4.7,\n    "TotalReviews": 157,\n    "Included": [\n      "Hotel pick up and drop off",\n      "English tour guide",\n      "Vietnamese Local Food",\n      "Air-conditioned vehicle",\n      "Entrance fees of Marble mountains - Am Phu Cave",\n      "A bottle 

In [3]:
test_cases = [

    "Tôi muốn xem review về tour PRbeRPTXGXPd ."
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)


🧑‍💻 User: Tôi muốn xem review về tour PRbeRPTXGXPd .
{'messages': [{'role': 'user', 'content': 'Tôi muốn xem review về tour PRbeRPTXGXPd .'}]}
🔧 Tool: fetch_attraction_reviews_tool
📥 Args: {'id': 'PRbeRPTXGXPd'}
Tool Output: [{'type': 'text', 'text': '{\n  "source": "booking_com15_rapidapi",\n  "id": "PRbeRPTXGXPd",\n  "good_reviews": [\n    {\n      "content": "Great tour. Having a guide is really worthwhile, and in our case, Nam, who was great. Hadn\'t realised how much there is beyond the bridge! Definitely recommended if in Da Nang.",\n      "numericRating": 5\n    },\n    {\n      "content": "Better than portrayed on social media",\n      "numericRating": 5\n    },\n    {\n      "content": "This was the best trip, and was very glad it was guided by LY she was amazing, we would have got lost without her, she was so friendly and very knowledgeable and has very good English, and gave us all good tips for the rest of our holiday, thanks for a great day Ly and we loved BaNa Hills",\n 